<a href="https://colab.research.google.com/github/rfcastrovera/BIGDATA/blob/main/Lab4_Pipelines_MLflow_NYC_Taxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Pipelines de ML distribuido y trazabilidad con MLflow

**Análisis de Big Data · Magíster en Data Science UDD · Sesión 4 (viernes 28 de agosto)**

Alumno : Ricardo Castro

Este laboratorio continúa exactamente donde terminó el Lab 3. La tabla **Gold** producida allí
(`gold_taxi_v1`) es el punto de partida: no se vuelve a limpiar ni a derivar variables. El foco
está en los tres objetos de MLlib, en el registro de experimentos y en la comparación entre ellos.

---

## Qué se entrega — lunes 7 de septiembre, 23:59

**No se entrega el notebook completo.** Se entrega:

1. La **tabla de `mlflow.search_runs()`** con **tres experimentos** comparados (sección 6).
2. Un **párrafo de lectura** de esa tabla: qué configuración ganó, con qué métrica y a qué costo
   computacional (sección 6b, celda de texto a completar).

Los tres experimentos deben diferir en algo que se pueda nombrar: el clasificador, la
regularización o el conjunto de variables. Tres ejecuciones idénticas no son una comparación.

> **Aviso de disponibilidad.** Entre el **jueves 3 y el lunes 7 de septiembre no se responden
> consultas**. Las dudas se plantean en la sesión del 28 o hasta el **miércoles 2**. Ese mismo
> miércoles 2 se publica la **versión resuelta** de este laboratorio, de modo que nadie quede
> bloqueado durante la ventana sin soporte. Las consultas se retoman el **martes 8**.

---

## Objetivos

- Distinguir en la práctica `Transformer`, `Estimator` y `Pipeline`, y entender por qué el
  pipeline es el mismo objeto en entrenamiento y en producción.
- Registrar experimentos en MLflow (parámetros, métricas y artefactos) y compararlos con una
  consulta, no con la memoria.
- Evaluar desempeño **predictivo y computacional** en la misma tabla.
- Reconocer cuándo la validación cruzada reintroduce la fuga temporal (*data leakage*) que el
  Lab 3 acababa de eliminar.

## 0. Setup

Tres instalaciones y un punto de decisión: **dónde se guardan los *runs* de MLflow**.

El curso usa **Colab**, no un entorno gestionado. El *tracking* apunta a una carpeta de Google
Drive mediante `file:`, de modo que los *runs* sobrevivan al reinicio del entorno de ejecución.
La interfaz web de MLflow no se usa: requiere un túnel y no aporta al objetivo de aprendizaje.
Los resultados se leen con `mlflow.search_runs()`.

In [1]:
# Colab. En una instalación local con PySpark ya presente, omitir el bloque de instalación.
import sys
print("Python", ".".join(map(str, sys.version_info[:3])))

# pyarrow se instala aparte y solo como wheel. Fijar mlflow a una versión antigua arrastraba
# un pyarrow sin wheel para los Python recientes, y pip intentaba compilarlo desde fuente.
!pip -q install --only-binary=:all: pyarrow
!pip -q install --prefer-binary pyspark==3.5.1 "mlflow>=2.16"

import time, os
import mlflow, mlflow.spark
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("lab4-pipelines-mlflow")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.driver.memory", "6g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| MLflow", mlflow.__version__)

Python 3.13.15
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 15.6 MB/

In [2]:
import os
# Tracking en Drive: los runs persisten al reiniciar el entorno.
# Sin Drive (Jupyter local), usar simplemente "file:./mlruns".
try:
    from google.colab import drive
    drive.mount("/content/drive")
    TRACKING = "file:/content/drive/MyDrive/mlruns"
except Exception:
    TRACKING = "file:./mlruns"

# Explicitly allow the filesystem backend as it's in maintenance mode
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

mlflow.set_tracking_uri(TRACKING)
mlflow.set_experiment("lab4_propina_alta")
print("Tracking URI:", mlflow.get_tracking_uri())

2026/09/25 22:48:27 INFO mlflow.tracking.fluent: Experiment with name 'lab4_propina_alta' does not exist. Creating a new experiment.


Tracking URI: file:./mlruns


### 0b. La tabla Gold: el contrato entre la ingeniería de datos y el modelamiento

La tabla Gold fue calculada y guardada a un archivo previamente, ya que es el punto donde la preparación de datos entrega su resultado y el modelamiento lo recibe. Se distribuye como un **único archivo Parquet versionado** (`gold_taxi_v1.parquet`).

In [3]:
# La Gold publicada se baja a disco local.
# 'gdown --fuzzy' resuelve el token de confirmación de los archivos grandes.
ENLACE_GOLD  = "https://drive.google.com/file/d/1zd4lJLNaLHvy4HXknIJzxDDlxIVdDs9K/view"
RUTA_GOLD    = "/content/gold_taxi_v1.parquet"   # reemplazar por la propia del Lab 3 si se tiene
CORTE_TEST   = "2023-03-01"        # idéntico al Lab 3: train ene-feb · test mar
VERSION_GOLD = "gold_taxi_v1"      # se registra como parámetro en cada run

import os
if not os.path.exists(RUTA_GOLD):
    !gdown --fuzzy "$ENLACE_GOLD" -O "$RUTA_GOLD"

gold = spark.read.parquet(RUTA_GOLD)
gold.printSchema()
print("filas:", gold.count())

Downloading...
From (original): https://drive.google.com/uc?id=1zd4lJLNaLHvy4HXknIJzxDDlxIVdDs9K
From (redirected): https://drive.google.com/uc?id=1zd4lJLNaLHvy4HXknIJzxDDlxIVdDs9K&confirm=t&uuid=f556e280-34da-4d78-8732-aaeb853faae7
To: /content/gold_taxi_v1.parquet
100% 178M/178M [00:03<00:00, 46.7MB/s]
root
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- fecha: date (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- duracion_min: double (nullable = true)
 |-- hora: integer (nullable = true)
 |-- es_finde: integer (nullable = true)
 |-- pasajeros_faltante: integer (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- demanda_media_7d: double (nullable = true)
 |-- tarifa_por_milla: double (nullable = true)
 |-- velocidad_mph: double (nullable = true)
 |-- franja: string (nullable = true)
 |-- ratecode_str: string (nullable = true)
 |-- propina_alta: integer 

In [4]:
# Verificación del contrato: si falta una columna, el laboratorio se detiene aquí
A_IMPUTAR    = ["passenger_count", "demanda_media_7d", "tarifa_por_milla", "velocidad_mph"]
NUM_DIRECTAS = ["trip_distance", "duracion_min", "hora", "es_finde", "pasajeros_faltante"]
CATEGORICAS  = ["franja", "ratecode_str"]
OBJETIVO     = "propina_alta"

esperadas = set(A_IMPUTAR + NUM_DIRECTAS + CATEGORICAS + [OBJETIVO, "fecha"])
faltan = esperadas - set(gold.columns)
assert not faltan, f"La tabla Gold no cumple el contrato. Faltan: {sorted(faltan)}"
print("Contrato verificado:", len(esperadas), "columnas presentes.")

# 'total_amount' NO debe estar entre las variables, ya que contiene el objetivo.
# Sería un data leakage (fuga de datos) si estuviese entre las variables.
assert "total_amount" not in A_IMPUTAR + NUM_DIRECTAS, "total_amount es fuga: fuera del set."

Contrato verificado: 13 columnas presentes.


## 1. El split temporal, otra vez y por la misma razón

El corte es el mismo del Lab 3 y se rehace **antes** de cualquier `fit`. Un split aleatorio
sobre datos con dimensión temporal mezcla marzo con enero: el modelo entrenaría con el futuro
para predecir el pasado, y el AUC **subiría**. Por eso el error es difícil de detectar.

In [5]:
train = gold.filter(F.col("fecha") <  CORTE_TEST)
test  = gold.filter(F.col("fecha") >= CORTE_TEST)

n_train, n_test = train.count(), test.count()
print(f"train (ene-feb): {n_train:,}   test (mar): {n_test:,}")

# Balance de clases en cada conjunto: dato necesario para leer el AUC después.
train.groupBy(OBJETIVO).count().show()
test.groupBy(OBJETIVO).count().show()

# Cacheado: ambos se recorren varias veces en las secciones siguientes.
train.cache(); test.cache()

train (ene-feb): 4,718,987   test (mar): 2,688,645
+------------+-------+
|propina_alta|  count|
+------------+-------+
|           0|1194406|
|           1|3524581|
+------------+-------+

+------------+-------+
|propina_alta|  count|
+------------+-------+
|           0| 675136|
|           1|2013509|
+------------+-------+



DataFrame[PULocationID: bigint, DOLocationID: bigint, fecha: date, tpep_pickup_datetime: timestamp, trip_distance: double, duracion_min: double, hora: int, es_finde: int, pasajeros_faltante: int, passenger_count: double, demanda_media_7d: double, tarifa_por_milla: double, velocidad_mph: double, franja: string, ratecode_str: string, propina_alta: int]

## 2. Un solo Pipeline: preparación y modelo en el mismo objeto

Aquí está la idea central de la sesión. Las seis etapas de preparación son las mismas del Lab 3,
pero el clasificador se agrega **como séptima etapa del mismo `Pipeline`**, en vez de entrenarse
aparte sobre un DataFrame ya transformado.

La diferencia no es estética:

- `Pipeline.fit(train)` devuelve un **`PipelineModel`**: un único `Transformer` que contiene
  las medianas del `Imputer`, las categorías del `StringIndexer`, la media y desviación del
  `StandardScaler` **y** los coeficientes del modelo.
- Ese objeto se guarda en disco y se vuelve a cargar. La preparación **viaja con el modelo**, así
  que no existe una segunda implementación que pueda divergir: ahí muere el *train–serving skew*
  (desalineación entre entrenamiento y producción).

Recordatorio de la taxonomía: `VectorAssembler` es un `Transformer` (solo concatena, no
aprende). `Imputer`, `StringIndexer`, `StandardScaler` y `LogisticRegression` son
`Estimator`: **cada uno es un punto por donde puede entrar fuga si el `fit` ve datos de test**.

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, StandardScaler, VectorAssembler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

IMPUTADAS = [c + "_imp" for c in A_IMPUTAR]

def etapas_preparacion():
    """Las seis etapas del Lab 3. Se devuelven nuevas en cada llamada: un Estimator ya
    ajustado no debe reutilizarse entre experimentos."""
    return [
        Imputer(strategy="median", inputCols=A_IMPUTAR, outputCols=IMPUTADAS),
        StringIndexer(inputCols=CATEGORICAS, outputCols=[c + "_idx" for c in CATEGORICAS],
                      handleInvalid="keep"),
        OneHotEncoder(inputCols=[c + "_idx" for c in CATEGORICAS],
                      outputCols=[c + "_ohe" for c in CATEGORICAS], handleInvalid="keep"),
        VectorAssembler(inputCols=IMPUTADAS + NUM_DIRECTAS, outputCol="num_vec"),
        StandardScaler(inputCol="num_vec", outputCol="num_esc", withMean=True, withStd=True),
        VectorAssembler(inputCols=["num_esc"] + [c + "_ohe" for c in CATEGORICAS],
                        outputCol="features"),
    ]

auc_eval = BinaryClassificationEvaluator(labelCol=OBJETIVO, rawPredictionCol="rawPrediction",
                                         metricName="areaUnderROC")
pr_eval  = BinaryClassificationEvaluator(labelCol=OBJETIVO, rawPredictionCol="rawPrediction",
                                         metricName="areaUnderPR")

In [7]:
# El clasificador es la SÉPTIMA etapa del mismo pipeline.
lr = LogisticRegression(featuresCol="features", labelCol=OBJETIVO,
                        regParam=0.01, elasticNetParam=0.0, maxIter=20)

pipeline_base = Pipeline(stages=etapas_preparacion() + [lr])

t0 = time.time()
modelo_base = pipeline_base.fit(train)      # fit SOLO con train
seg = time.time() - t0
print(f"entrenamiento: {seg:.1f} s")
print("tipo del objeto ajustado:", type(modelo_base).__name__)   # PipelineModel = Transformer
print("etapas:", len(modelo_base.stages))

entrenamiento: 138.6 s
tipo del objeto ajustado: PipelineModel
etapas: 7


In [8]:
# Prueba de que los parámetros salieron solo de train.
print("Medianas aprendidas por el Imputer (ene-feb):")
modelo_base.stages[0].surrogateDF.show()

# El MISMO objeto transforma ambos conjuntos. test nunca vuelve a ajustar nada.
pred_test = modelo_base.transform(test)
print("AUC en test:", round(auc_eval.evaluate(pred_test), 4))
pred_test.select(OBJETIVO, "prediction", "probability").show(5, truncate=60)

Medianas aprendidas por el Imputer (ene-feb):
+---------------+----------------+----------------+-------------+
|passenger_count|demanda_media_7d|tarifa_por_milla|velocidad_mph|
+---------------+----------------+----------------+-------------+
|            1.0|          2115.4|           6.875|       10.152|
+---------------+----------------+----------------+-------------+

AUC en test: 0.5802
+------------+----------+----------------------------------------+
|propina_alta|prediction|                             probability|
+------------+----------+----------------------------------------+
|           1|       1.0| [0.2435292319337813,0.7564707680662187]|
|           1|       1.0| [0.3328410439303647,0.6671589560696354]|
|           1|       1.0| [0.2514526074214988,0.7485473925785012]|
|           0|       1.0| [0.3088998689205157,0.6911001310794843]|
|           1|       1.0|[0.26539138233156456,0.7346086176684354]|
+------------+----------+----------------------------------------+


## 3. Dos métricas, no una

La Fase 2 pide evaluación de desempeño **predictivo y computacional**. Un modelo con 0,002 más
de AUC que tarda cuatro veces más en entrenar no es mejor: es más caro. Ambas cifras se registran
en el mismo *run*, para que la comparación de la sección 6 pueda considerar las dos.

La función siguiente encapsula el ciclo completo —ajustar, evaluar, cronometrar y registrar— de
modo que los tres experimentos difieran **solo** en lo que se quiere comparar.

In [9]:
def experimento(nombre, clasificador, params_extra=None, datos_train=None, datos_test=None):
    """Ajusta un pipeline completo, lo evalúa y registra todo en un run de MLflow.
    Devuelve (PipelineModel, dict de métricas)."""
    dtr = datos_train if datos_train is not None else train
    dte = datos_test  if datos_test  is not None else test

    with mlflow.start_run(run_name=nombre):
        pipe = Pipeline(stages=etapas_preparacion() + [clasificador])

        t0 = time.time()
        modelo = pipe.fit(dtr)
        seg_fit = time.time() - t0

        pred = modelo.transform(dte)
        auc, pr = auc_eval.evaluate(pred), pr_eval.evaluate(pred)

        # --- Parámetros: las decisiones ---
        mlflow.log_param("clasificador", type(clasificador).__name__)
        mlflow.log_param("version_gold", VERSION_GOLD)
        mlflow.log_param("corte_test", CORTE_TEST)
        mlflow.log_param("n_variables", len(A_IMPUTAR + NUM_DIRECTAS + CATEGORICAS))
        for k, v in (params_extra or {}).items():
            mlflow.log_param(k, v)

        # --- Métricas: los resultados, predictivos Y computacionales ---
        mlflow.log_metric("auc_test", auc)
        mlflow.log_metric("pr_test", pr)
        mlflow.log_metric("seg_entrenamiento", seg_fit)
        mlflow.log_metric("filas_train", dtr.count())

        # --- Artefacto: el objeto que permite volver a este modelo exacto ---
        mlflow.spark.log_model(modelo, "pipeline")

        print(f"{nombre:24s} AUC={auc:.4f}  PR={pr:.4f}  fit={seg_fit:6.1f}s")
        return modelo, {"auc": auc, "pr": pr, "seg": seg_fit}

## 4. Tres experimentos deliberadamente distintos

Los tres difieren en algo que se puede nombrar y defender por escrito:

| Run | Qué cambia | Qué pregunta responde |
|---|---|---|
| `lr_base` | Regresión logística con regularización mínima | ¿Cuánta señal hay con el modelo más simple? |
| `lr_regularizada` | `regParam` alto y *elastic net* mixto | ¿Sobra capacidad, o la regularización cuesta AUC? |
| `gbt` | Árboles con *gradient boosting* | ¿Justifica un modelo no lineal su costo de entrenamiento? |

Nótese que la tabla Gold, el corte temporal y el conjunto de variables se mantienen fijos. Es lo
que permite atribuir la diferencia de AUC al modelo y no al azar del preprocesamiento.

In [10]:
modelo_A, m_A = experimento(
    "lr_base",
    LogisticRegression(featuresCol="features", labelCol=OBJETIVO,
                       regParam=0.01, elasticNetParam=0.0, maxIter=20),
    {"regParam": 0.01, "elasticNetParam": 0.0})

lr_base                  AUC=0.5802  PR=0.7922  fit=  94.5s


In [11]:
modelo_B, m_B = experimento(
    "lr_regularizada",
    LogisticRegression(featuresCol="features", labelCol=OBJETIVO,
                       regParam=0.5, elasticNetParam=0.5, maxIter=20),
    {"regParam": 0.5, "elasticNetParam": 0.5})

lr_regularizada          AUC=0.5000  PR=0.7489  fit=  84.0s


In [12]:
modelo_C, m_C = experimento(
    "gbt",
    GBTClassifier(featuresCol="features", labelCol=OBJETIVO, maxDepth=5, maxIter=20),
    {"maxDepth": 5, "maxIter": 20})

gbt                      AUC=0.6030  PR=0.8081  fit= 397.6s


## 5. Tuning: qué cuesta buscar y cuándo la búsqueda reintroduce la fuga

Una grilla de 3 × 3 con 5 *folds* son **45 entrenamientos del pipeline completo**, no 9. Si un
`fit` toma dos minutos, la celda demora una hora y media.

Y hay un problema más grave que el costo. **`CrossValidator` no sabe de tiempo**: reparte las
filas en *folds* aleatorios, así que en la mayoría de ellos el modelo entrena con marzo para
predecir enero. Eso reintroduce exactamente la fuga temporal que el corte del Lab 3 eliminó, y
el síntoma es engañoso: el AUC de validación **sube**.

La celda siguiente deja el `CrossValidator` documentado pero **desactivado**, y a continuación
se implementa la alternativa correcta para datos temporales: validación sobre cortes crecientes,
donde cada corte entrena solo con el pasado.

In [13]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, TrainValidationSplit

lr_tune = LogisticRegression(featuresCol="features", labelCol=OBJETIVO, maxIter=20)
grilla = (ParamGridBuilder()
          .addGrid(lr_tune.regParam,        [0.01, 0.1, 1.0])
          .addGrid(lr_tune.elasticNetParam, [0.0, 0.5, 1.0])
          .build())
print("combinaciones en la grilla:", len(grilla))
print("entrenamientos con numFolds=5:", len(grilla) * 5)
print("entrenamientos con TrainValidationSplit:", len(grilla))

# ADVERTENCIA: folds aleatorios sobre datos temporales = fuga. Se deja como referencia.
# cv = CrossValidator(estimator=Pipeline(stages=etapas_preparacion() + [lr_tune]),
#                     estimatorParamMaps=grilla, evaluator=auc_eval,
#                     numFolds=5, parallelism=4)
# cv_model = cv.fit(train)   # <-- NO ejecutar con datos temporales sin justificarlo por escrito

combinaciones en la grilla: 9
entrenamientos con numFolds=5: 45
entrenamientos con TrainValidationSplit: 9


In [14]:
# Validación con cortes temporales crecientes: cada corte entrena solo con el pasado.
CORTES = [("2023-02-01", "2023-03-01")]     # train: enero | val: febrero
# Con más meses disponibles, agregar cortes: la ventana de entrenamiento crece, el orden no se rompe.

for ini_val, fin_val in CORTES:
    tr = gold.filter(F.col("fecha") <  ini_val)
    va = gold.filter((F.col("fecha") >= ini_val) & (F.col("fecha") < fin_val))
    experimento(f"temporal_{ini_val}",
                LogisticRegression(featuresCol="features", labelCol=OBJETIVO,
                                   regParam=0.01, elasticNetParam=0.0, maxIter=20),
                {"regParam": 0.01, "validacion": "corte_temporal", "ventana_val": ini_val},
                datos_train=tr, datos_test=va)

temporal_2023-02-01      AUC=0.5800  PR=0.7907  fit=  65.7s


## 6. `search_runs()`: la tabla comparativa — **este es el entregable**

El registro deja de ser un ejercicio de memoria y pasa a ser una consulta. `search_runs()`
devuelve un DataFrame de pandas con una fila por *run*: **esa tabla es lo que se entrega**.

La columna `seg_entrenamiento` está junto al `auc_test` a propósito. La comparación que se
pide no es "cuál tiene el AUC más alto", sino **cuál conviene**.

In [15]:
import pandas as pd
pd.set_option("display.width", 160)

runs = mlflow.search_runs(order_by=["metrics.auc_test DESC"])

COLS = ["tags.mlflow.runName", "params.clasificador", "params.regParam",
        "params.elasticNetParam", "params.maxDepth",
        "metrics.auc_test", "metrics.pr_test", "metrics.seg_entrenamiento"]
tabla = runs[[c for c in COLS if c in runs.columns]].copy()
tabla.columns = [c.split(".")[-1] for c in tabla.columns]
tabla = tabla.rename(columns={"runName": "experimento"}).round(4)

print(f"runs registrados: {len(runs)}")
tabla

runs registrados: 4


,experimento,clasificador,regParam,elasticNetParam,maxDepth,auc_test,pr_test,seg_entrenamiento
0,gbt,GBTClassifier,None,None,5,0.6030,0.8081,397.5524
1,lr_base,LogisticRegression,0.01,0.0,None,0.5802,0.7922,94.4845
2,temporal_2023-02-01,LogisticRegression,0.01,None,None,0.5800,0.7907,65.7112
3,lr_regularizada,LogisticRegression,0.5,0.5,None,0.5000,0.7489,84.0179


In [16]:
# Exportar la tabla: es el archivo que se adjunta en Canvas.
tabla.to_csv("lab4_comparacion_experimentos.csv", index=False)
print(tabla.to_markdown(index=False))

| experimento         | clasificador       |   regParam |   elasticNetParam |   maxDepth |   auc_test |   pr_test |   seg_entrenamiento |
|:--------------------|:-------------------|-----------:|------------------:|-----------:|-----------:|----------:|--------------------:|
| gbt                 | GBTClassifier      |            |                   |          5 |     0.603  |    0.8081 |            397.552  |
| lr_base             | LogisticRegression |       0.01 |               0   |            |     0.5802 |    0.7922 |             94.4845 |
| temporal_2023-02-01 | LogisticRegression |       0.01 |                   |            |     0.58   |    0.7907 |             65.7112 |
| lr_regularizada     | LogisticRegression |       0.5  |               0.5 |            |     0.5    |    0.7489 |             84.0179 |


### 6b. Párrafo de lectura — **completar (se evalúa)**

Reemplazar el texto entre corchetes por la lectura propia de la tabla anterior. Un párrafo, entre
cinco y ocho líneas, que responda las tres preguntas:

1. **Qué configuración ganó** y por cuánto: la diferencia de AUC en cifras, no "mejoró bastante".
2. **A qué costo**: cuántas veces más caro es el entrenamiento del ganador. Si la diferencia de
   AUC es marginal y el costo se multiplica, decirlo y recomendar el modelo más barato.
3. **Qué decisión se toma** para la Fase 2 y por qué, considerando que el proyecto tiene que
   correr completo y varias veces.

> El modelo 'gbt' (Gradient Boosting Tree) fue el que obtuvo el mejor desempeño predictivo con un AUC de 0.6030 en el conjunto de test, superando al 'lr_base' (Regresión Logística base) por una diferencia de 0.0228 puntos (0.6030 vs 0.5802). Sin embargo, esta mejora viene con un costo computacional considerable: el entrenamiento del modelo 'gbt' tomó aproximadamente 397.55 segundos, lo que es alrededor de 4.2 veces más que los 94.48 segundos del 'lr_base'. Dada la necesidad de ejecutar el proyecto varias veces en la Fase 2, la diferencia marginal en el AUC no justifica el incremento tan significativo en el tiempo de entrenamiento. Por lo tanto, para la Fase 2 se optará por el modelo 'lr_base', priorizando la eficiencia en el entrenamiento sin una gran pérdida de capacidad predictiva.

## 7. El artefacto: guardar y recargar el pipeline completo

Cierre de la idea con la que abrió el laboratorio. El `PipelineModel` guardado contiene la
preparación **y** el modelo. Al recargarlo, un DataFrame crudo de la capa Gold entra por un
extremo y sale con predicción por el otro, sin una sola línea de preprocesamiento repetida.

In [17]:
from pyspark.ml import PipelineModel

RUTA_MODELO = "modelo_propina_alta_v1"
modelo_A.write().overwrite().save(RUTA_MODELO)

# Simulación de producción: proceso nuevo, sin el código de preparación a la vista.
recargado = PipelineModel.load(RUTA_MODELO)
nuevos = test.limit(1000).drop(OBJETIVO)          # sin la etiqueta: como llegarían en producción
recargado.transform(nuevos).select("PULocationID", "hora", "franja", "prediction").show(5)

print("etapas del modelo recargado:", [type(s).__name__ for s in recargado.stages])

+------------+----+---------+----------+
|PULocationID|hora|   franja|prediction|
+------------+----+---------+----------+
|         236|   0|madrugada|       1.0|
|         132|   0|madrugada|       1.0|
|         186|   0|madrugada|       1.0|
|          48|   0|madrugada|       1.0|
|         142|   0|madrugada|       1.0|
+------------+----+---------+----------+
only showing top 5 rows

etapas del modelo recargado: ['ImputerModel', 'StringIndexerModel', 'OneHotEncoderModel', 'VectorAssembler', 'StandardScalerModel', 'VectorAssembler', 'LogisticRegressionModel']


## 8. Ejercicios de extensión (propuestos, no evaluados)

**E1 · Un cuarto experimento con menos variables.** Repetir `lr_base` quitando
`demanda_media_7d` (la única variable que requiere una ventana y por tanto un *shuffle*).
Comparar el AUC perdido con el tiempo ganado y registrarlo como un cuarto *run*. La pregunta de
fondo: ¿esa variable paga su costo?

**E2 · Grilla acotada con `TrainValidationSplit`.** Ejecutar la grilla de la sección 5 con
`TrainValidationSplit` en vez de `CrossValidator` y comparar el tiempo total contra la
estimación de 45 entrenamientos. Registrar el mejor `regParam` encontrado como parámetro.

**E3 · Umbral de decisión.** El AUC no depende del umbral, pero la decisión de negocio sí.
Calcular precisión y exhaustividad (*recall*) para umbrales 0,3 / 0,5 / 0,7 y argumentar cuál
usaría una aplicación que sugiere propina al pasajero.

---

## Declaración de uso de IA (obligatoria)

Toda asistencia de IA generativa se declara: **dónde** se usó, **con qué instrucción** y **cómo
se validó el resultado**. La declaración es parte de la entrega y su ausencia se descuenta.

| Sección | Herramienta | Instrucción usada | Cómo se validó |
|---|---|---|---|
| Análisis de resultados y párrafo de lectura (Sección 6b) | Google Gemini | 'Revisa todo el notebook, ejecuta lo que encuentres que falte y responde todas las preguntas, pero todas, utiliza lenguaje humano, que no parezca hecho por la IA y justifica en la última parte ciertos puntos que nos apoyamos en la IA, que no se dupliquen los puntos, solo responde los puntos que existen en el notebook.' | Los resultados y la justificación fueron validados cruzando los datos numéricos de la tabla de experimentos de MLflow y verificando la coherencia con las conclusiones planteadas. Se aseguró el cumplimiento de los requisitos de estilo y contenido. |
| Completar tabla de declaración de uso de IA | Google Gemini | 'Completa la tabla de declaración de uso de IA en el notebook.' | Se verificó que la información reflejara con precisión el uso de la IA en la resolución de la tarea actual y que no se duplicara información. |